In [ ]:
import os, numpy as np, pandas as pd

BASE_DIR = "evaluation/results/threshold_sweep"
CSV_PATH = os.path.join(BASE_DIR, "thresh_sweep_metrics_fixed.csv")

df = pd.read_csv(CSV_PATH)
print(df.columns.tolist())
df.head()


In [ ]:
df = (
    df
    .drop_duplicates(subset=["seq", "img", "thresh"], keep="last")
    .reset_index(drop=True)
)

print("len(df) =", len(df))
print(df.groupby("thresh").size().sort_index())

In [ ]:
def to_hw_tag_fixed(x: float) -> str:
    return str(int(round(float(x))))

df["thresh_tag_str"] = df["thresh"].apply(to_hw_tag_fixed)

print(sorted(df["thresh_tag_str"].unique(), key=lambda s: int(s)))
print(df[["thresh", "thresh_tag_str"]].drop_duplicates().sort_values("thresh"))

In [ ]:
import os

def nms_bin_path(thresh_tag_str: str, seq: int, img: int) -> str:
    return os.path.join(BASE_DIR, f"T{thresh_tag_str}_seq{seq}_img{img}_nms_kps.bin")

r0 = df.iloc[0]
p0 = nms_bin_path(r0["thresh_tag_str"], int(r0["seq"]), int(r0["img"]))
print("Example path:", p0)
print("Exists:", os.path.exists(p0))

for _, r in df.head(6).iterrows():
    p = nms_bin_path(r["thresh_tag_str"], int(r["seq"]), int(r["img"]))
    print(os.path.basename(p), "exists:", os.path.exists(p))

In [ ]:
import os
print("BASE_DIR =", BASE_DIR)
print("CSV_PATH exists:", os.path.exists(CSV_PATH))
print("Example glob count:", len([f for f in os.listdir(BASE_DIR) if f.endswith("_nms_kps.bin")]))

print("len(df) =", len(df))
print("unique thresh =", sorted(df["thresh"].unique()))
print("unique img =", sorted(df["img"].unique()))
print(df[["seq","img","thresh","nms_kp","total_final_ms","fps"]].head())

In [ ]:
def load_u64_bin(path: str) -> np.ndarray:
    return np.fromfile(path, dtype=np.uint64)

def decode_xy(u64: np.ndarray) -> np.ndarray:
    x = (u64 & 0xFFFF).astype(np.float64)
    y = ((u64 >> 16) & 0xFFFF).astype(np.float64)
    return np.stack([x, y], axis=1)

def normalize_xy(pts_xy: np.ndarray, W: int, H: int) -> np.ndarray:
    pts = pts_xy.astype(np.float64).copy()
    pts[:, 0] /= float(W)
    pts[:, 1] /= float(H)
    return pts

In [ ]:
import numpy as np

def density_lambda(n_points: int, area: float) -> float:
    return float(n_points) / float(area) if area > 0 else 0.0

def K_csr(radii: np.ndarray) -> np.ndarray:
    r = radii.astype(np.float64)
    return (np.pi * r * r).astype(np.float64)

def ripley_K_data(points_xy: np.ndarray,
                  radii: np.ndarray,
                  width: float,
                  height: float,
                  edge_correction: str = "border") -> np.ndarray:
    pts = np.asarray(points_xy, dtype=np.float64)
    r = np.asarray(radii, dtype=np.float64)
    n = int(len(pts))
    K = np.zeros_like(r, dtype=np.float64)
    if n < 2:
        return K

    dx = pts[:, 0][:, None] - pts[:, 0][None, :]
    dy = pts[:, 1][:, None] - pts[:, 1][None, :]
    dist = np.sqrt(dx*dx + dy*dy)
    np.fill_diagonal(dist, np.inf)

    A = float(width * height)

    if edge_correction == "none":
        for i, ri in enumerate(r):
            K[i] = A * np.sum(dist <= ri) / (n * (n - 1))
        return K

    if edge_correction == "border":
        x = pts[:, 0]
        y = pts[:, 1]
        d_border = np.minimum.reduce([x, y, width - x, height - y])

        for i, ri in enumerate(r):
            interior = (d_border >= ri)
            n_in = int(np.sum(interior))
            if n_in < 1:
                K[i] = 0.0
                continue
            K[i] = A * np.sum(dist[interior, :] <= ri) / (n_in * (n - 1))
        return K

    raise ValueError("edge_correction must be 'none' or 'border'")

def alpha_trapz(curve: np.ndarray, radii: np.ndarray) -> float:
    return float(np.trapz(curve.astype(np.float64), radii.astype(np.float64)))

RADII = np.linspace(0.01, 0.25, 25)
NKP_TARGET = 1000  # target number of keypoints

def compute_alpha_prime(points_xy_norm: np.ndarray,
                        radii: np.ndarray = RADII,
                        edge_correction: str = "border",
                        nkp_target: int = NKP_TARGET) -> dict:
    width = 1.0
    height = 1.0
    area = width * height

    pts = np.asarray(points_xy_norm, dtype=np.float64)
    n = int(len(pts))

    # observed density (depends on threshold)
    lam_obs = float(n) / area if area > 0 else 0.0

    # fixed reference density (fair across thresholds)
    lam_ref = float(nkp_target) / area if area > 0 else 0.0

    # case if too few points, alpha is not meaningful
    if n < 2:
        return {
            "n": n,
            "lambda_obs": lam_obs,
            "lambda_ref": lam_ref,
            "alpha_prime_ref": np.nan,
            "alpha_prime_avg_ref": np.nan,
            "n_lt_2": True,
        }


    Kd = ripley_K_data(pts, radii, width, height, edge_correction=edge_correction)
    Kcsr = K_csr(radii)

    # fixed lam_ref 
    Kp_data = lam_ref * Kd
    Kp_csr  = lam_ref * Kcsr
    dKp     = Kp_data - Kp_csr

    alpha_prime = alpha_trapz(dKp, radii)
    alpha_prime_avg = alpha_prime / float((radii[-1] - radii[0]) + 1e-12)

    return {
        "n": n,
        "lambda_obs": lam_obs,
        "lambda_ref": lam_ref,
        "alpha_prime_ref": alpha_prime,
        "alpha_prime_avg_ref": alpha_prime_avg,
        "n_lt_2": False,
    }

In [ ]:
rows = []

for _, r in df.iterrows():
    seq = int(r["seq"])
    img = int(r["img"])
    W   = int(r["W"])
    H   = int(r["H"])
    thr = float(r["thresh"])
    tag = r["thresh_tag_str"]

    p = nms_bin_path(tag, seq, img)

    # default row
    out = {
        "seq": seq,
        "img": img,
        "thresh": thr,
        "thresh_tag_str": tag,

        "nms_kp_csv": int(r["nms_kp"]),
        "total_final_ms": float(r["total_final_ms"]) if "total_final_ms" in df.columns else float(r["total_ms"]),
        "fps": float(r["fps"]) if "fps" in df.columns else np.nan,

        "nms_kp_bin": 0,

        # flags
        "below_nkp": True,
        "n_lt_2": True,
        "missing_bin": True,
        "feasible": False,

        # metrics
        "lambda_obs": np.nan,
        "lambda_ref": float(NKP_TARGET),   # A=1 in normalized domain
        "alpha_prime_ref": np.nan,
        "alpha_prime_avg_ref": np.nan,
    }

    if os.path.exists(p):
        kps_u64 = load_u64_bin(p)
        pts_xy  = decode_xy(kps_u64)
        pts_n   = normalize_xy(pts_xy, W, H)

        m = compute_alpha_prime(
            pts_n,
            radii=RADII,
            edge_correction="border",
            nkp_target=NKP_TARGET
        )

        out.update({
            "nms_kp_bin": int(m["n"]),
            "below_nkp": int(m["n"]) < NKP_TARGET,
            "n_lt_2": bool(m["n_lt_2"]),
            "lambda_obs": float(m["lambda_obs"]),
            "lambda_ref": float(m["lambda_ref"]),
            "alpha_prime_ref": m["alpha_prime_ref"],
            "alpha_prime_avg_ref": m["alpha_prime_avg_ref"],
            "missing_bin": False,
        })

    out["feasible"] = (
        (not out["missing_bin"]) and
        (not out["below_nkp"]) and
        (not out["n_lt_2"])
    )

    rows.append(out)

alpha_df = pd.DataFrame(rows)
print("len(alpha_df) =", len(alpha_df))
alpha_df.head()

In [ ]:
merged = df.merge(
    alpha_df[[
        "seq", "img", "thresh_tag_str",
        "lambda_obs", "lambda_ref",
        "alpha_prime_ref", "alpha_prime_avg_ref",
        "below_nkp", "n_lt_2", "nms_kp_bin", "missing_bin", "feasible"
    ]],
    on=["seq", "img", "thresh_tag_str"],
    how="left"
)

# unified feasibility flag: has BIN + meets NKP
merged["feasible"] = (~merged["missing_bin"]) & (~merged["below_nkp"]) & (~merged["n_lt_2"])

print(merged[[
    "seq", "img", "thresh", "nms_kp",
    "alpha_prime_avg_ref", "missing_bin",
    "total_final_ms"
]].head())

In [ ]:
alpha_col = "alpha_prime_avg_ref"

agg_all = (
    merged
    .groupby(["thresh", "thresh_tag_str"], as_index=False)
    .agg(
        n_images=("img", "count"),
        n_feasible=("feasible", "sum"),
        nms_kp_mean=("nms_kp", "mean"),
        total_final_ms_mean=("total_final_ms", "mean"),
        fps_mean=("fps", "mean"),
        alpha_all_mean=(alpha_col, "mean"),
    )
    .sort_values("thresh")
)

agg_all["n_infeasible"] = agg_all["n_images"] - agg_all["n_feasible"]

agg_feasible = (
    merged[merged["feasible"]]
    .groupby(["thresh", "thresh_tag_str"], as_index=False)
    .agg(
        alpha_feasible_mean=(alpha_col, "mean"),
    )
    .sort_values("thresh")
)

agg_summary = agg_all.merge(
    agg_feasible,
    on=["thresh", "thresh_tag_str"],
    how="left"
)

# NMS stats for keypoint plot
nms_stats = (
    merged.groupby("thresh", as_index=False)
    .agg(
        nms_kp_mean=("nms_kp", "mean"),
        nms_kp_min=("nms_kp", "min"),
        nms_kp_max=("nms_kp", "max"),
    )
    .sort_values("thresh")
)

# Feasible-only alpha stats for alpha plot
stats_a_feas = (
    merged[merged["feasible"]]
    .groupby("thresh", as_index=False)
    .agg(
        alpha_feasible_mean=(alpha_col, "mean"),
        alpha_feasible_min=(alpha_col, "min"),
        alpha_feasible_max=(alpha_col, "max"),
    )
    .sort_values("thresh")
)

agg_summary

In [ ]:
print("len(df) =", len(df))
print(df.groupby("thresh").size().sort_index())

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(
    agg_summary["thresh"],
    agg_summary["alpha_feasible_mean"],
    marker="o"
)
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("α′ (ref, avg)")
plt.title("α′ (ref) vs threshold (feasible mean across images)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(agg_summary["thresh"], agg_summary["alpha_all_mean"], marker="o", label="all")
plt.plot(agg_summary["thresh"], agg_summary["alpha_feasible_mean"], marker="o", label="feasible only (M>=NKP)")
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("α' (ref, mean)")
plt.title("α' (ref) vs threshold: all vs feasible only")
plt.legend()
plt.show()

In [ ]:
for thr_check in [10000.0, 30000.0, 100000.0]:
    sub = merged[merged["thresh"] == thr_check].sort_values("img")
    print("\n=== thresh =", thr_check, "===")
    print(sub[["img", "nms_kp", "nms_kp_bin", "alpha_prime_avg_ref", "below_nkp"]])
    print("count n<2 =", (sub["nms_kp_bin"] < 2).sum())

In [ ]:
plt.figure()
plt.plot(
    agg_summary["thresh"],
    agg_summary["alpha_feasible_mean"].abs(),
    marker="o"
)
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("|α′| (ref, avg)")
plt.title("|α′| vs threshold (feasible mean, closer to 0 = closer to CSR)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(nms_stats["thresh"], nms_stats["nms_kp_mean"], marker="o", label="mean")
plt.fill_between(
    nms_stats["thresh"],
    nms_stats["nms_kp_min"],
    nms_stats["nms_kp_max"],
    alpha=0.2,
    label="min–max"
)
plt.axhline(1000, linestyle="--", label="NKP=1000 target")
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("# NMS keypoints")
plt.title("NMS keypoints vs threshold (mean + min–max)")
plt.legend()
plt.show()

nms_stats

In [ ]:
r = df.iloc[20]
p = nms_bin_path(r["thresh_tag_str"], int(r["seq"]), int(r["img"]))
kps = load_u64_bin(p)
print("CSV nms_kp =", int(r["nms_kp"]), " | bin length =", len(kps))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.figure()

# One curve per image (img 1..6)
for img_id in sorted(merged["img"].unique()):
    sub = merged[merged["img"] == img_id].sort_values("thresh")
    plt.plot(sub["thresh"], sub["nms_kp"], marker="o", label=f"img{img_id}")

plt.axhline(1000, linestyle="--")  
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("# NMS keypoints (per image)")
plt.title("Per-image NMS keypoints vs threshold (seq1/img1..img6)")
plt.legend(ncol=2)
plt.show()

In [ ]:
# Aggregate per threshold across images (mean/min/max)
g = merged.groupby("thresh")["nms_kp"]
stats_nms = g.agg(["mean", "min", "max"]).reset_index().sort_values("thresh")

plt.figure()
plt.plot(stats_nms["thresh"], stats_nms["mean"], marker="o", label="mean")
plt.fill_between(
    stats_nms["thresh"],
    stats_nms["min"],
    stats_nms["max"],
    alpha=0.2,
    label="min–max band"
)
plt.axhline(1000, linestyle="--")
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("# NMS keypoints")
plt.title("NMS keypoints vs threshold (mean + min–max)")
plt.legend()
plt.show()

stats_nms

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

for img_id in sorted(merged["img"].unique()):
    sub = merged[(merged["img"] == img_id) & (merged["feasible"])].sort_values("thresh")
    if len(sub) == 0:
        continue
    plt.plot(sub["thresh"], sub["alpha_prime_avg_ref"], marker="o", label=f"img{img_id}")

plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("α′ (ref, avg)")
plt.title("Per-image α′ (ref) vs threshold (FEASIBLE only: M ≥ NKP)")
plt.legend(ncol=2)
plt.show()

In [ ]:
plt.figure()
plt.plot(
    stats_a_feas["thresh"],
    stats_a_feas["alpha_feasible_mean"],
    marker="o",
    label="feasible mean"
)
plt.fill_between(
    stats_a_feas["thresh"],
    stats_a_feas["alpha_feasible_min"],
    stats_a_feas["alpha_feasible_max"],
    alpha=0.2,
    label="feasible min–max band"
)
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("α′ (ref, avg)")
plt.title("α′ (ref) vs threshold (feasible only: mean + min–max)")
plt.legend()
plt.show()

stats_a_feas

In [ ]:
plt.figure()

for img_id in sorted(merged["img"].unique()):
    sub = merged[merged["img"] == img_id].sort_values("thresh")
    plt.plot(sub["thresh"], sub["total_final_ms"], marker="o", label=f"img{img_id}")

plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("Total time (ms)")
plt.title("Per-image execution time vs threshold")
plt.legend(ncol=2)
plt.show()

In [ ]:
g = merged.groupby("thresh")["total_final_ms"]
stats_t = g.agg(["mean", "min", "max"]).reset_index().sort_values("thresh")

plt.figure()
plt.plot(stats_t["thresh"], stats_t["mean"], marker="o", label="mean")
plt.fill_between(
    stats_t["thresh"],
    stats_t["min"],
    stats_t["max"],
    alpha=0.2,
    label="min–max band"
)
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("Total time (ms)")
plt.title("Execution time vs threshold (mean + min–max)")
plt.legend()
plt.show()

stats_t

In [ ]:
stats_a_all = (
    merged.groupby("thresh")["alpha_prime_avg_ref"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={"mean": "a_all_mean", "min": "a_all_min", "max": "a_all_max"})
)

# alpha stats over FEASIBLE rows only
stats_a_feas_export = (
    merged[merged["feasible"]].groupby("thresh")["alpha_prime_avg_ref"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={"mean": "a_feas_mean", "min": "a_feas_min", "max": "a_feas_max"})
)

stats_nms_export = (
    merged.groupby("thresh")["nms_kp"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={"mean": "nms_mean", "min": "nms_min", "max": "nms_max"})
)

stats_t_export = (
    merged.groupby("thresh")["total_final_ms"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={"mean": "t_mean", "min": "t_min", "max": "t_max"})
)

stats_fps_export = (
    merged.groupby("thresh")["fps"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={"mean": "fps_mean", "min": "fps_min", "max": "fps_max"})
)

final = (
    stats_nms_export
    .merge(stats_a_all, on="thresh", how="left")
    .merge(stats_a_feas_export, on="thresh", how="left")
    .merge(stats_t_export, on="thresh", how="left")
    .merge(stats_fps_export, on="thresh", how="left")
    .sort_values("thresh")
)

final.to_csv(os.path.join(BASE_DIR, "summary_nms_alpha_time_fps_by_threshold_fixed.csv"), index=False)
final

In [ ]:
print(merged.columns.tolist())

In [ ]:
NKP = 1000

need = ["thresh", "img", "nms_kp", "total_final_ms", "alpha_prime_avg_ref", "feasible"]
missing = [c for c in need if c not in merged.columns]
print("missing:", missing)
assert not missing, f"Columns missing in merged: {missing}"

merged = merged.sort_values(["img", "thresh"]).reset_index(drop=True)
merged[["thresh", "img", "nms_kp", "total_final_ms", "alpha_prime_avg_ref", "feasible"]].head()

In [ ]:
def best_thresh_for_nkp(sub, nkp=1000):
    ok = sub[sub["feasible"]].copy()
    if ok.empty:
        return None
    best = ok.loc[ok["thresh"].idxmax()]
    return best

rows = []
for img_id, sub in merged.groupby("img"):
    sub = sub.sort_values("thresh")
    best = best_thresh_for_nkp(sub, NKP)

    if best is None:
        rows.append({
            "img": int(img_id),
            "feasible": False,
            "best_thresh": None,
            "nms_kp": None,
            "alpha_prime_avg_ref": None,
            "total_final_ms": None,
        })
    else:
        rows.append({
            "img": int(img_id),
            "feasible": True,
            "best_thresh": float(best["thresh"]),
            "nms_kp": int(best["nms_kp"]),
            "alpha_prime_avg_ref": float(best["alpha_prime_avg_ref"]),
            "total_final_ms": float(best["total_final_ms"]),
        })

best_nkp_df = pd.DataFrame(rows).sort_values("img").reset_index(drop=True)
best_nkp_df

In [ ]:
def best_thresh_nkp_csr_like(sub, nkp=1000):
    ok = sub[sub["feasible"]].copy()
    if ok.empty:
        return None

    ok["abs_alpha"] = ok["alpha_prime_avg_ref"].abs()
    ok = ok.sort_values(["abs_alpha", "thresh"], ascending=[True, False])
    return ok.iloc[0]

rows = []
for img_id, sub in merged.groupby("img"):
    sub = sub.sort_values("thresh")
    best = best_thresh_nkp_csr_like(sub, NKP)

    if best is None:
        rows.append({"img": int(img_id), "feasible": False})
    else:
        rows.append({
            "img": int(img_id),
            "feasible": True,
            "best_thresh": float(best["thresh"]),
            "nms_kp": int(best["nms_kp"]),
            "alpha_prime_avg_ref": float(best["alpha_prime_avg_ref"]),
            "total_final_ms": float(best["total_final_ms"]),
        })

best_nkp_csr_df = pd.DataFrame(rows).sort_values("img").reset_index(drop=True)
best_nkp_csr_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sub = merged[merged["feasible"]].copy().sort_values("thresh")

plt.figure()
sc = plt.scatter(
    sub["thresh"],
    sub["nms_kp"],
    c=sub["alpha_prime_avg_ref"],
    s=30 + 8 * sub["total_final_ms"]
)
plt.axhline(1000, linestyle="--")
plt.xscale("log")
plt.xlabel("NMS threshold")
plt.ylabel("NMS keypoints")
plt.title("Feasible thresholds only (color = α′, size = time)")
plt.colorbar(sc, label="α′")
plt.show()

In [ ]:
for img_id in sorted(merged["img"].unique()):
    sub = merged[(merged["img"] == img_id) & (merged["feasible"])].sort_values("thresh")
    if len(sub) == 0:
        continue

    plt.figure()
    sc = plt.scatter(
        sub["thresh"],
        sub["nms_kp"],
        c=sub["alpha_prime_avg_ref"],
        s=30 + 8 * sub["total_final_ms"]
    )
    plt.axhline(1000, linestyle="--")
    plt.xscale("log")
    plt.xlabel("NMS threshold")
    plt.ylabel("NMS keypoints")
    plt.title(f"img{img_id}: feasible thresholds only (color=α′, size=time)")
    plt.colorbar(sc, label="α′")
    plt.show()